# New_RAG v2 - Ground Truth 기반 공구명 표준화

## 핵심 설계 변경
- **지식 베이스(Vector DB)**: `standard_tool_names_ground_truth_v2.csv`의 `standard_name` 목록만 사용
  - ❌ `original_name → standard_name` 매핑은 RAG에서 참조하지 않음 (컨닝 방지)
  - ✅ `standard_name` 160개 목록, `brand` 18개, `power_source` 8개 목록만 참조
- **입력 데이터**: `ToolNameData.csv` (3352개 original_name 정제 대상)
- **평가**: ground_truth_v2의 원본→표준명 매핑으로 정확도 계산
- **모델 순서** (크기 오름차순): deepseek-r1:8b → granite4.1:8b → qwen3.5:9b → 이후 순차 진행

In [1]:
import ollama
import json
import pandas as pd
import re
from tqdm import tqdm
import numpy as np
import warnings

try:
    import faiss
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics import accuracy_score
    print("모든 라이브러리 로드 완료.")
except ImportError as e:
    print(f"[오류] 필수 라이브러리 없음: {e}")
    print("설치 명령: pip install sentence-transformers faiss-cpu scikit-learn ollama")

warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'ollama'

In [ ]:
# ===== 설정 변수 =====

# 파일 경로
TOOL_DATA_PATH    = './ToolNameData.csv'
GROUND_TRUTH_PATH = './standard_tool_names_ground_truth_v2.csv'
OUTPUT_DIR        = './'

# 모델 설정 (용량 오름차순으로 순차 실행)
MODEL_NAME      = 'deepseek-r1:8b'      # 5.2GB - 가장 작음
MODEL_SAVE_NAME = 'deepseek-r1_8b'

# RAG 설정
K_VALUE  = 5   # 상위 k개 후보 검색
TEST_N   = 10  # 테스트 행 수 (전체 실행 시 None)

print(f"모델: {MODEL_NAME}")
print(f"테스트 행 수: {TEST_N}")

In [ ]:
# ===== 설치된 ollama 모델 목록 확인 =====
try:
    models = ollama.list()
    model_list = [(m.model, getattr(m, 'size', None)) for m in models.models]
    # 크기 기준 오름차순 정렬
    model_list.sort(key=lambda x: x[1] if x[1] else float('inf'))
    print("=== 설치된 모델 (크기 오름차순) ===")
    for name, size in model_list:
        size_gb = f"{size/1e9:.1f}GB" if size else "Unknown"
        print(f"  {name:55s} {size_gb}")
except Exception as e:
    print(f"ollama 연결 실패: {e}")
    print("WSL에서 'ollama serve'가 실행 중인지 확인하세요.")

In [ ]:
# ===== 데이터 로드 =====

# 1. 정제 대상: ToolNameData
df_tool = pd.read_csv(TOOL_DATA_PATH)
print(f"ToolNameData 로드 완료: {df_tool.shape}")
print(df_tool.head(3))

print()

# 2. 정답지: standard_tool_names_ground_truth_v2
df_gt = pd.read_csv(GROUND_TRUTH_PATH)
print(f"Ground Truth 로드 완료: {df_gt.shape}")
print(df_gt.head(3))

In [ ]:
# ===== RAG 지식 베이스 구축 =====
# 중요: original_name 컬럼은 RAG에서 절대 사용하지 않음
#        standard_name / brand / power_source 목록만 사용

STANDARD_NAMES  = df_gt['standard_name'].dropna().unique().tolist()
BRANDS          = df_gt['brand'].dropna().unique().tolist()
POWER_SOURCES   = df_gt['power_source'].dropna().unique().tolist()

print(f"표준 공구명 종류: {len(STANDARD_NAMES)}개")
print(f"브랜드 목록: {BRANDS}")
print(f"전원 방식 목록: {POWER_SOURCES}")

In [ ]:
# ===== FAISS Vector DB 구축 (standard_name 목록 기반) =====
def build_standard_name_db(standard_names,
                            model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'):
    """표준 공구명 목록을 임베딩하여 FAISS 인덱스 생성"""
    print(f"Vector DB 구축 시작 (표준 공구명 {len(standard_names)}개 임베딩)...")
    embed_model = SentenceTransformer(model_name)
    vectors = embed_model.encode(standard_names, show_progress_bar=True)
    dim = vectors.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(vectors.astype('float32'))
    print(f"FAISS Vector DB 구축 완료 (d={dim}, 항목={len(standard_names)}개)")
    return index, embed_model

faiss_index, embed_model = build_standard_name_db(STANDARD_NAMES)

In [ ]:
# ===== 프롬프트 템플릿 =====
PROMPT_TEMPLATE = """
당신은 한국어 하드웨어 공구 이름 분류 전문가입니다.
정제되지 않은 공구 이름을 분석하여 아래 목록에서 가장 적합한 표준 정보를 추출하세요.

[표준 공구명 후보 (유사도 상위 {k}개)]
{standard_name_candidates}

[브랜드 목록] (공구 이름에 브랜드가 없으면 null)
{brand_list}

[전원 방식 목록] (없으면 null)
{power_source_list}

규칙:
- standard_name은 반드시 위 '표준 공구명 후보' 중 하나를 선택하세요.
- brand는 '브랜드 목록' 중 하나이거나 null이어야 합니다.
- power_source는 '전원 방식 목록' 중 하나이거나 null이어야 합니다.
- specification은 규격, 크기, 기타 사양 정보입니다 (없으면 null).

JSON 형식으로만 응답하세요:
{{
  "standard_name": "표준 공구명",
  "brand": "브랜드 또는 null",
  "power_source": "전원 방식 또는 null",
  "specification": "사양 또는 null"
}}

입력 공구 이름: '{tool_name}'
출력:
"""

print("프롬프트 템플릿 정의 완료")

In [ ]:
# ===== RAG 워크플로우 함수 =====
def run_rag_v2(tool_name, faiss_index, embed_model, standard_names, brands, power_sources,
               model_name, k=5):
    """
    tool_name: 정제할 원본 공구명
    faiss_index: standard_name 목록으로 만든 FAISS 인덱스
    standard_names: 표준 공구명 목록
    """
    # 1. Retrieve: 유사 표준 공구명 k개 검색
    query_vec = embed_model.encode([str(tool_name).strip()]).astype('float32')
    distances, indices = faiss_index.search(query_vec, k)
    candidates = [standard_names[i] for i in indices[0]]
    
    # 2. Augment: 프롬프트 구성
    prompt = PROMPT_TEMPLATE.format(
        k=k,
        standard_name_candidates='\n'.join(f'  - {c}' for c in candidates),
        brand_list=', '.join(brands),
        power_source_list=', '.join(power_sources),
        tool_name=str(tool_name).strip()
    )
    
    # 3. Generate: LLM 추론
    try:
        response = ollama.generate(
            model=model_name,
            prompt=prompt,
            format='json',
            stream=False,
            options={"temperature": 0}
        )
        response_text = response.get('response', '{}')
        
        # deepseek-r1 등 <think> 태그 제거
        response_text = re.sub(r'<think>.*?</think>', '', response_text, flags=re.DOTALL).strip()
        
        # JSON 파싱
        parsed = json.loads(response_text)
        
        # standard_name이 후보 목록에 없으면 가장 유사한 후보로 교체
        predicted = parsed.get('standard_name', '')
        if predicted not in standard_names:
            parsed['standard_name'] = candidates[0]  # fallback: top-1 후보
            parsed['fallback_used'] = True
        else:
            parsed['fallback_used'] = False
        
        return {'status': 'success', 'data': parsed, 'rag_candidates': candidates}
    
    except Exception as e:
        return {'status': 'error', 'message': str(e), 'rag_candidates': candidates}

print("RAG 함수 정의 완료")

In [ ]:
# ===== 메인 실행 (테스트: 10개 행) =====

# 테스트 데이터 선택
df_test = df_tool.head(TEST_N).copy() if TEST_N else df_tool.copy()
print(f"처리 대상: {len(df_test)}개 행")
print(df_test[['Original_Name']].to_string())

print(f"\n--- RAG 정제 시작 (모델: {MODEL_NAME}) ---")

In [ ]:
# ===== 처리 실행 =====
processed_results = []

for tool_name in tqdm(df_test['Original_Name'], desc=f"{MODEL_NAME} 처리 중"):
    result = run_rag_v2(
        tool_name, faiss_index, embed_model,
        STANDARD_NAMES, BRANDS, POWER_SOURCES,
        MODEL_NAME, k=K_VALUE
    )
    
    entry = {
        'Input_Name': tool_name,
        'status': result['status'],
        'rag_top1': result['rag_candidates'][0] if result.get('rag_candidates') else None,
        'rag_candidates': str(result.get('rag_candidates', []))
    }
    
    if result['status'] == 'success':
        data = result['data']
        entry['standard_name']  = data.get('standard_name')
        entry['brand']          = data.get('brand')
        entry['power_source']   = data.get('power_source')
        entry['specification']  = data.get('specification')
        entry['fallback_used']  = data.get('fallback_used', False)
    else:
        entry['error_message'] = result.get('message')
    
    processed_results.append(entry)

df_results = pd.DataFrame(processed_results)
print("\n처리 완료!")
print(df_results[['Input_Name', 'standard_name', 'brand', 'power_source', 'fallback_used']])

In [ ]:
# ===== 평가 (ground_truth_v2 기반) =====
# 평가용 매핑: ground_truth_v2의 original_name → standard_name
# (RAG에서 사용한 게 아닌, 평가 목적으로만 사용)

gt_map = df_gt.set_index('original_name')['standard_name'].to_dict()

# Input_Name이 ground_truth_v2에 있는 경우 정답 부여
df_results['gt_standard_name'] = df_results['Input_Name'].map(gt_map)

# 정/오답 판단
def check_correct(row):
    if pd.isna(row['gt_standard_name']):
        return None  # 정답 없음 (평가 불가)
    return str(row.get('standard_name', '')).strip() == str(row['gt_standard_name']).strip()

df_results['is_correct'] = df_results.apply(check_correct, axis=1)

# 평가 결과 출력
evaluatable = df_results[df_results['gt_standard_name'].notna()]
print("\n=== 평가 결과 ===")
print(f"전체 처리: {len(df_results)}개")
print(f"정답 존재 (평가 가능): {len(evaluatable)}개")

if len(evaluatable) > 0:
    correct_count = (evaluatable['is_correct'] == True).sum()
    accuracy = correct_count / len(evaluatable)
    print(f"정확도: {correct_count}/{len(evaluatable)} = {accuracy:.2%}")
    print()
    print(evaluatable[['Input_Name', 'standard_name', 'gt_standard_name', 'is_correct']].to_string())
else:
    print("테스트 10개 중 ground_truth_v2에 있는 항목이 없습니다.")
    print("(ToolNameData 전체 실행 시 457개 정답과 비교 가능)")

print()
print("=== 전체 결과 ===")
display_cols = ['Input_Name', 'standard_name', 'brand', 'power_source', 'specification',
                'rag_top1', 'fallback_used', 'gt_standard_name', 'is_correct']
print(df_results[[c for c in display_cols if c in df_results.columns]].to_string())

In [ ]:
# ===== 결과 저장 =====
save_path = OUTPUT_DIR + f'New_RAG_v2_{MODEL_SAVE_NAME}_test{TEST_N}.csv'
df_results.to_csv(save_path, index=False, encoding='utf-8-sig')
print(f"결과 저장 완료: {save_path}")
print(df_results)

---
## 전체 데이터 실행 가이드

테스트 성공 후 전체 3352개 실행:
1. 위 셀에서 `TEST_N = None` 으로 변경
2. `MODEL_NAME`을 순서대로 변경하며 실행:
   - `deepseek-r1:8b` (5.2GB)
   - `granite4.1:8b` (5.3GB)
   - `qwen3.5:9b` (6.6GB)
   - `gemma4:e4b` (9.6GB)
   - `granite4.1:30b` (17GB)
   - `qwen3.5:27b` (17GB)
   - `deepseek-r1:32b` (19GB)
   - `gemma4:31b` (19GB)
   - `hf.co/LGAI-EXAONE/EXAONE-4.5-33B-GGUF:Q4_K_M` (22GB)

결과 평가는 `New_RAG_Evaluation2.ipynb` 패턴 참고